In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# ========== READ ==========
book3_path = r"C:\Users\pljh0187\Desktop\Senushi\Book3.csv"
df = pd.read_csv(book3_path)
df.columns = df.columns.str.strip()

print(f"Total compounds loaded: {len(df)}")

# ========== CHECK H ATOMS ==========
no_H = []
has_H = []
invalid = []

for _, row in df.iterrows():
    smiles = str(row['SMILES']).strip()

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            invalid.append(row)
            continue

        # Add explicit hydrogens to count all H atoms
        mol_with_H = Chem.AddHs(mol)
        h_count = sum(1 for atom in mol_with_H.GetAtoms() if atom.GetAtomicNum() == 1)

        if h_count == 0:
            no_H.append(row)
        else:
            has_H.append(row)

    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        invalid.append(row)

# ========== RESULTS ==========
df_has_H   = pd.DataFrame(has_H)
df_no_H    = pd.DataFrame(no_H)
df_invalid = pd.DataFrame(invalid)

print(f"\n  Has H atoms (usable)    : {len(df_has_H)}")
print(f"  No H atoms  (unusable)  : {len(df_no_H)}")
print(f"  Invalid SMILES          : {len(df_invalid)}")

# ========== SAVE ==========
out_path = r"C:\Users\pljh0187\Desktop\Senushi\Book3_H_check.xlsx"

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    df_has_H.to_excel(writer,   sheet_name='Has_H_atoms',   index=False)
    df_no_H.to_excel(writer,    sheet_name='No_H_atoms',    index=False)
    if len(df_invalid) > 0:
        df_invalid.to_excel(writer, sheet_name='Invalid_SMILES', index=False)

print(f"\nSaved to: {out_path}")

# ========== PREVIEW NO H ==========
if len(df_no_H) > 0:
    print(f"\nCompounds with NO H atoms:")
    for _, row in df_no_H.iterrows():
        print(f"  ID={row['ID']}  |  {row['Name']}  |  {row['SMILES']}")
else:
    print("\nAll compounds have at least one H atom — none to exclude.")

Total compounds loaded: 502

  Has H atoms (usable)    : 495
  No H atoms  (unusable)  : 7
  Invalid SMILES          : 0

Saved to: C:\Users\pljh0187\Desktop\Senushi\Book3_H_check.xlsx

Compounds with NO H atoms:
  ID=2  |  3-nitro-1,2,4-triazol-5-one  |  [O-][N+](=O)C1=N\C(=O)\N=N1
  ID=133  |  hexachloroethane  |  ClC(Cl)(Cl)C(Cl)(Cl)Cl
  ID=188  |  hexachlorobenzene  |  c1(c(c(c(c(c1Cl)Cl)Cl)Cl)Cl)Cl
  ID=213  |  octafluoronaphthalene  |  Fc2c1c(F)c(F)c(F)c(F)c1c(F)c(F)c2F
  ID=265  |  decafluorobiphenyl  |  Fc1c(c(F)c(F)c(F)c1F)c2c(F)c(F)c(F)c(F)c2F
  ID=271  |  tetrabromomethane  |  C(Br)(Br)(Br)Br
  ID=436  |  octachlorodibenzofuran  |  o1c3c(c2c1c(c(c(c2Cl)Cl)Cl)Cl)c(c(c(c3Cl)Cl)Cl)Cl


Adding 7 new compounds to the list

In [2]:
import pandas as pd
from rdkit import Chem

# ========== READ ==========
book3_path = r"C:\Users\pljh0187\Desktop\Senushi\Book3.csv"
basf_path  = r"C:\Users\pljh0187\Desktop\Senushi\basf_cleaned.csv"

df_book3 = pd.read_csv(book3_path)
df_basf  = pd.read_csv(basf_path)

df_book3.columns = df_book3.columns.str.strip()
df_basf.columns  = df_basf.columns.str.strip()

df_basf = df_basf.rename(columns={'names': 'Name', 'smiles': 'SMILES', 'MW': 'MW'})

df_book3['Name']   = df_book3['Name'].astype(str).str.strip()
df_book3['SMILES'] = df_book3['SMILES'].astype(str).str.strip()
df_basf['Name']    = df_basf['Name'].astype(str).str.strip()
df_basf['SMILES']  = df_basf['SMILES'].astype(str).str.strip()

print(f"Book3 original : {len(df_book3)} compounds")

# ========== STEP 1 — REMOVE THE 7 NO-H COMPOUNDS BY ID ==========
ids_to_remove = ['2', '133', '188', '213', '265', '271', '436']

df_book3['ID'] = df_book3['ID'].astype(str).str.strip()
df_book3_clean = df_book3[~df_book3['ID'].isin(ids_to_remove)].copy()

print(f"After removing 7 no-H compounds : {len(df_book3_clean)} compounds")
print(f"Need to add                      : {502 - len(df_book3_clean)} compounds")

# ========== STEP 2 — H ATOM CHECK HELPER ==========
def has_h_atoms(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        mol_with_H = Chem.AddHs(mol)
        return sum(1 for atom in mol_with_H.GetAtoms() if atom.GetAtomicNum() == 1) > 0
    except:
        return False

# ========== STEP 3 — FILTER BASF CANDIDATES ==========
existing_names  = set(df_book3_clean['Name'].str.lower())
existing_smiles = set(df_book3_clean['SMILES'])

rejected_no_H       = []
rejected_dup_name   = []
rejected_dup_smiles = []
candidates          = []

for _, row in df_basf.iterrows():
    name   = str(row['Name']).strip()
    smiles = str(row['SMILES']).strip()

    # Rule 1 — must have H atoms
    if not has_h_atoms(smiles):
        rejected_no_H.append(name)
        continue

    # Rule 2 — no duplicate name
    if name.lower() in existing_names:
        rejected_dup_name.append(name)
        continue

    # Rule 3 — no duplicate SMILES
    if smiles in existing_smiles:
        rejected_dup_smiles.append(name)
        continue

    candidates.append(row)

print(f"\nRejected — no H atoms       : {len(rejected_no_H)}")
print(f"Rejected — duplicate name   : {len(rejected_dup_name)}")
print(f"Rejected — duplicate SMILES : {len(rejected_dup_smiles)}")
print(f"Candidates available        : {len(candidates)}")

# ========== STEP 4 — TAKE EXACTLY 7 ==========
needed = 502 - len(df_book3_clean)

if len(candidates) < needed:
    print(f"\nWARNING: only {len(candidates)} valid candidates available, need {needed}!")
else:
    print(f"\nTaking {needed} compounds from BASF")

df_new = pd.DataFrame(candidates[:needed]).copy()

# ========== STEP 5 — BUILD BOOK4 ==========
df_book4 = pd.concat(
    [df_book3_clean[['Name', 'SMILES', 'MW']],
     df_new[['Name', 'SMILES', 'MW']]],
    ignore_index=True
)

# Reassign all IDs cleanly from 001 to 502
df_book4.insert(0, 'ID', [str(i + 1).zfill(3) for i in range(len(df_book4))])

print(f"\nBook4 total compounds : {len(df_book4)}")

# ========== STEP 6 — FINAL DUPLICATE CHECK ==========
dup_smiles = df_book4[df_book4.duplicated(subset='SMILES', keep=False)]
dup_names  = df_book4[df_book4.duplicated(subset='Name',   keep=False)]

print(f"Duplicate SMILES in Book4 : {len(dup_smiles)}")
print(f"Duplicate Names  in Book4 : {len(dup_names)}")

if len(dup_smiles) > 0:
    print("\nDuplicate SMILES found:")
    print(dup_smiles[['ID', 'Name', 'SMILES']])

if len(dup_names) > 0:
    print("\nDuplicate Names found:")
    print(dup_names[['ID', 'Name', 'SMILES']])

# ========== SAVE ==========
out_path = r"C:\Users\pljh0187\Desktop\Senushi\Book4.csv"
df_book4.to_csv(out_path, index=False)
print(f"\nBook4 saved to: {out_path}")

# ========== PREVIEW ==========
print(f"\nRemoved from Book3:")
removed = df_book3[df_book3['ID'].isin(ids_to_remove)][['ID', 'Name', 'SMILES']]
print(removed.to_string(index=False))

print(f"\nNew 7 compounds added from BASF:")
print(df_book4.tail(7)[['ID', 'Name', 'SMILES', 'MW']].to_string(index=False))

Book3 original : 502 compounds
After removing 7 no-H compounds : 495 compounds
Need to add                      : 7 compounds


[12:10:51] SMILES Parse Error: syntax error while parsing: nan
[12:10:51] SMILES Parse Error: check for mistakes around position 2:
[12:10:51] nan
[12:10:51] ~^
[12:10:51] SMILES Parse Error: Failed parsing SMILES 'nan' for input: 'nan'
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 9 10 11 12 13
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 9
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13 14 15 16 17
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 11
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8 9 10 12 13 14 15 16 17
[12:10:51] Can'


Rejected — no H atoms       : 69
Rejected — duplicate name   : 668
Rejected — duplicate SMILES : 27
Candidates available        : 3180

Taking 7 compounds from BASF

Book4 total compounds : 502
Duplicate SMILES in Book4 : 0
Duplicate Names  in Book4 : 0

Book4 saved to: C:\Users\pljh0187\Desktop\Senushi\Book4.csv

Removed from Book3:
 ID                        Name                                            SMILES
  2 3-nitro-1,2,4-triazol-5-one                       [O-][N+](=O)C1=N\C(=O)\N=N1
133            hexachloroethane                            ClC(Cl)(Cl)C(Cl)(Cl)Cl
188           hexachlorobenzene                    c1(c(c(c(c(c1Cl)Cl)Cl)Cl)Cl)Cl
213       octafluoronaphthalene                Fc2c1c(F)c(F)c(F)c(F)c1c(F)c(F)c2F
265          decafluorobiphenyl        Fc1c(c(F)c(F)c(F)c1F)c2c(F)c(F)c(F)c(F)c2F
271           tetrabromomethane                                   C(Br)(Br)(Br)Br
436      octachlorodibenzofuran o1c3c(c2c1c(c(c(c2Cl)Cl)Cl)Cl)c(c(c(c3Cl)Cl)Cl)Cl

New 7 

[12:10:51] non-ring atom 0 marked aromatic
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13 14 15 16 17
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 2 3 4 6 8
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:10:51] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 26
